In [22]:
from pathlib import Path
import duckdb

database_path = Path("../database/football.duckdb")

connection = duckdb.connect(
    str(database_path),
    read_only=False
)

query = """
SELECT
    type,
    COUNT(*) AS total
FROM events
GROUP BY type
ORDER BY total DESC
"""

df_types = connection.execute(query).fetchdf()

display(df_types)

,type,total
0,Pass,33772
1,Ball Receipt*,29564
2,Carries,26926
3,Pressure,14267
4,Ball Recovery,3329
5,Duel,2118
6,Block,1453
7,Clearance,1191
8,Goal Keeper,1143
9,Foul Committed,898


In [23]:
query = """
SELECT
    COUNT(*) AS total_shots,
    COUNT(DISTINCT match_id) AS matches
FROM events
WHERE LOWER(type) = 'shot'
"""

df_shots = connection.execute(query).fetchdf()

display(df_shots)

,total_shots,matches
0,886,34


In [24]:
query = """
SELECT
    match_id,
    COUNT(*) AS total_shots
FROM events
WHERE LOWER(type) = 'shot'
GROUP BY match_id
ORDER BY total_shots DESC
"""

df_shots_match = connection.execute(query).fetchdf()

display(df_shots_match)

,match_id,total_shots
0,4000040,37
1,3999822,37
2,3999844,36
3,3999983,36
4,3999878,36
5,3999995,31
6,3999964,31
7,3999851,30
8,3999895,29
9,3999802,28


In [25]:
query = """
SELECT
    event_id,
    match_id,
    team_id,
    player_id,
    location
FROM events
WHERE LOWER(type) = 'shot'
LIMIT 20
"""

df_shot_locations = connection.execute(query).fetchdf()

display(df_shot_locations)

,event_id,match_id,team_id,player_id,location
0,9116c4b9-26a9-5919-ba65-2e05e085aab9,3999407,129,16238.0,"(107.6, 51.7)"
1,7fedbb8c-646d-5833-a9c4-64ff3aa48fef,3999407,164,32866.0,"(95.4, 29.9)"
2,ba930b7e-83b1-5d71-a2ca-4137223103a5,3999407,129,8776.0,"(103.6, 50.1)"
3,fa119e74-2949-5682-a3ff-33d75d2bca50,3999407,129,134854.0,"(96.5, 39.8)"
4,f8845ad4-d6a7-51fc-9e37-18aa27c1870b,3999407,129,392153.0,"(93.8, 35.7)"
5,2761d763-7064-5e90-b817-67709757ae09,3999407,129,8776.0,"(111.2, 31.5)"
6,ec23a4a4-9cdf-515c-b5a7-cf5b55e2d1a6,3999407,164,8434.0,"(104.3, 48.2)"
7,8410e435-3f31-55a3-966a-d493115660e4,3999407,129,144884.0,"(111.9, 50.3)"
8,55221385-bc94-510f-a39b-6613e6ac14f8,3999407,129,5177.0,"(101.0, 35.4)"
9,89389423-3218-53e6-8a20-ecff5711e40b,3999407,129,16238.0,"(96.2, 56.5)"


In [26]:
from pathlib import Path
import pandas as pd

# Récupérer les CSV StatsBomb
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_directory = project_root / "data" / "raw"

files = sorted(
    raw_directory.glob(
        "Ligue 1 2025_2026 AS Monaco p*.csv"
    )
)

print("Fichiers trouvés :", len(files))

# Charger et fusionner les fichiers
raw_df = pd.concat(
    [pd.read_csv(file) for file in files],
    ignore_index=True
)

print("Nombre total d'événements :", len(raw_df))

# Identifier les colonnes intéressantes
keywords = [
    "shot",
    "pass",
    "location",
    "player",
    "team",
    "id"
]

relevant_columns = [
    column
    for column in raw_df.columns
    if any(keyword in column.lower() for keyword in keywords)
]

print("\n===== COLONNES DISPONIBLES =====")

for column in relevant_columns:
    print(column)


Fichiers trouvés : 2


C:\Users\USER\AppData\Local\Temp\ipykernel_11400\2052788333.py:22: DtypeWarning: Columns (60,97,99,237,238,241) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(file) for file in files],
C:\Users\USER\AppData\Local\Temp\ipykernel_11400\2052788333.py:22: DtypeWarning: Columns (237,238,240,241,243,247) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(file) for file in files],


Nombre total d'événements : 136332

===== COLONNES DISPONIBLES =====
id
match_id
event_type_id
possession_team_id
possession_team_name
play_pattern_id
team_id
team_name
player_id
player_name
player_position_id
player_position_name
location_x
location_y
end_location_x
end_location_y
end_location_z
substituted_player_id
substituted_player_name
pass_recipient_id
pass_recipient_name
pass_length
pass_angle
pass_height_id
pass_height_name
pass_cross
pass_cut_back
pass_switch
assisted_shot_id
key_pass_id
outcome_id
type_id
card_id
body_part_id
technique_id
position_id
early_video_end
late_video_start
freeze_frame_player_id
freeze_frame_player_name
freeze_frame_player_position_id
freeze_frame_player_position_name
freeze_frame_teammate
formation_player_id
formation_player_name
formation_position_id
shot_shot_assist
shot_goal_assist
pass_cluster_id
pass_cluster_label
pass_cluster_probability
pass_success_probability
shot_execution_xg
shot_execution_xg_uplift
gk_shot_stopping_xg_suppression
playe

In [27]:
# Extraire les tirs de toutes les équipes
shots_df = raw_df[
    raw_df["event_type_name"] == "Shot"
].copy()

print("Nombre de tirs :", len(shots_df))

# Colonnes liées aux tirs
shot_columns = [
    column
    for column in shots_df.columns
    if "shot" in column.lower()
    or column in ["id", "match_id", "team_name", "player_name"]
]

display(
    shots_df[shot_columns].head(15)
)


Nombre de tirs : 13620


,id,match_id,team_name,player_name,assisted_shot_id,shot_shot_assist,shot_goal_assist,shot_execution_xg,shot_execution_xg_uplift,gk_shot_stopping_xg_suppression
170,9116c4b9-26a9-5919-ba65-2e05e085aab9,3999407,AS Monaco,Jordan Teze,NaN,False,False,0.000000,-0.057619,NaN
213,7fedbb8c-646d-5833-a9c4-64ff3aa48fef,3999407,Le Havre,Issa Soumaré,NaN,False,False,0.041754,0.016550,0.044573
506,ba930b7e-83b1-5d71-a2ca-4137223103a5,3999407,AS Monaco,Takumi Minamino,NaN,False,False,0.127542,0.041294,0.057169
794,fa119e74-2949-5682-a3ff-33d75d2bca50,3999407,AS Monaco,Mika Miles Biereth,NaN,False,False,0.000000,-0.042027,NaN
1187,f8845ad4-d6a7-51fc-9e37-18aa27c1870b,3999407,AS Monaco,Lamine Camara,NaN,False,False,0.079693,0.055527,NaN
1347,2761d763-7064-5e90-b817-67709757ae09,3999407,AS Monaco,Takumi Minamino,NaN,False,False,0.036836,-0.011885,NaN
1519,ec23a4a4-9cdf-515c-b5a7-cf5b55e2d1a6,3999407,Le Havre,Younes Namli,NaN,False,False,0.103944,0.033445,NaN
1920,8410e435-3f31-55a3-966a-d493115660e4,3999407,AS Monaco,Maghnes Akliouche,NaN,False,False,0.000000,-0.106687,NaN
2127,55221385-bc94-510f-a39b-6613e6ac14f8,3999407,AS Monaco,Aleksandr Golovin,NaN,False,False,0.005208,-0.045034,NaN
2134,89389423-3218-53e6-8a20-ecff5711e40b,3999407,AS Monaco,Jordan Teze,NaN,False,False,0.022177,0.002449,NaN


In [28]:
# Identifier les passes associées à un tir
pass_columns = [
    column
    for column in raw_df.columns
    if "pass" in column.lower()
]

print("Colonnes liées aux passes :")

for column in pass_columns:
    print(column)

# Examiner les passes identifiées comme assists
if "pass_shot_assist" in raw_df.columns:

    assists_df = raw_df[
        raw_df["pass_shot_assist"] == True
    ]

    print(
        "\nNombre de passes préparant un tir :",
        len(assists_df)
    )

    display(assists_df.head(10))


Colonnes liées aux passes :
pass_recipient_id
pass_recipient_name
pass_length
pass_angle
pass_height_id
pass_height_name
pass_cross
pass_cut_back
pass_switch
key_pass_id
pass_cluster_id
pass_cluster_label
pass_cluster_probability
pass_success_probability


In [29]:
# Toutes les colonnes contenant xG, shot ou outcome
columns_to_check = [
    column for column in raw_df.columns
    if any(
        keyword in column.lower()
        for keyword in ["xg", "shot", "outcome", "body_part", "technique"]
    )
]

print("===== COLONNES DÉTAILLÉES =====")

for column in columns_to_check:
    print(column)

===== COLONNES DÉTAILLÉES =====
assisted_shot_id
statsbomb_xg
outcome_id
outcome_name
body_part_id
body_part_name
technique_id
technique_name
shot_shot_assist
shot_goal_assist
shot_execution_xg
shot_execution_xg_uplift
gk_save_difficulty_xg
gk_positioning_xg_suppression
gk_shot_stopping_xg_suppression


In [30]:
shots = raw_df[
    raw_df["event_type_name"] == "Shot"
].copy()

print("Nombre de tirs :", len(shots))

if "shot_execution_xg" in shots.columns:

    print("\n===== STATISTIQUES SHOT EXECUTION XG =====")

    display(
        shots["shot_execution_xg"].describe()
    )

    print(
        "Valeurs manquantes :",
        shots["shot_execution_xg"].isna().sum()
    )

print("\n===== EXEMPLE DE TIRS =====")

display(
    shots[
        ["id", "match_id", "team_name", "player_name",
         "shot_execution_xg"]
    ].head(10)
)

Nombre de tirs : 13620

===== STATISTIQUES SHOT EXECUTION XG =====


count    13620.000000
mean         0.111167
std          0.209326
min          0.000000
25%          0.000000
50%          0.017494
75%          0.106057
max          0.996852
Name: shot_execution_xg, dtype: float64

Valeurs manquantes : 0

===== EXEMPLE DE TIRS =====


,id,match_id,team_name,player_name,shot_execution_xg
170,9116c4b9-26a9-5919-ba65-2e05e085aab9,3999407,AS Monaco,Jordan Teze,0.000000
213,7fedbb8c-646d-5833-a9c4-64ff3aa48fef,3999407,Le Havre,Issa Soumaré,0.041754
506,ba930b7e-83b1-5d71-a2ca-4137223103a5,3999407,AS Monaco,Takumi Minamino,0.127542
794,fa119e74-2949-5682-a3ff-33d75d2bca50,3999407,AS Monaco,Mika Miles Biereth,0.000000
1187,f8845ad4-d6a7-51fc-9e37-18aa27c1870b,3999407,AS Monaco,Lamine Camara,0.079693
1347,2761d763-7064-5e90-b817-67709757ae09,3999407,AS Monaco,Takumi Minamino,0.036836
1519,ec23a4a4-9cdf-515c-b5a7-cf5b55e2d1a6,3999407,Le Havre,Younes Namli,0.103944
1920,8410e435-3f31-55a3-966a-d493115660e4,3999407,AS Monaco,Maghnes Akliouche,0.000000
2127,55221385-bc94-510f-a39b-6613e6ac14f8,3999407,AS Monaco,Aleksandr Golovin,0.005208
2134,89389423-3218-53e6-8a20-ecff5711e40b,3999407,AS Monaco,Jordan Teze,0.022177


In [31]:
# Événements possédant un identifiant de tir associé
assisted_events = raw_df[
    raw_df["assisted_shot_id"].notna()
].copy()

print(
    "Événements avec assisted_shot_id :",
    len(assisted_events)
)

print("\n===== TYPES D'ÉVÉNEMENTS =====")

display(
    assisted_events["event_type_name"].value_counts()
)

print("\n===== EXEMPLES =====")

display(
    assisted_events[
        [
            "id",
            "event_type_name",
            "team_name",
            "player_name",
            "assisted_shot_id",
            "shot_shot_assist",
            "shot_goal_assist"
        ]
    ].head(15)
)

Événements avec assisted_shot_id : 709

===== TYPES D'ÉVÉNEMENTS =====


event_type_name
Pass    679
Shot     30
Name: count, dtype: int64


===== EXEMPLES =====


,id,event_type_name,team_name,player_name,assisted_shot_id,shot_shot_assist,shot_goal_assist
166,cb5ab9e0-b82b-57fa-b216-8af4085ed558,Pass,AS Monaco,Aleksandr Golovin,9116c4b9-26a9-5919-ba65-2e05e085aab9,NaN,NaN
498,ab5fab03-ab33-51fe-a1b1-6eab4e37b771,Pass,AS Monaco,Maghnes Akliouche,ba930b7e-83b1-5d71-a2ca-4137223103a5,NaN,NaN
1343,6a936571-8b97-5b7c-9533-3cb03a638149,Pass,AS Monaco,Lamine Camara,2761d763-7064-5e90-b817-67709757ae09,NaN,NaN
1514,60c99230-5e76-5d61-9744-70575b2fc390,Pass,Le Havre,Arouna Sangante,ec23a4a4-9cdf-515c-b5a7-cf5b55e2d1a6,NaN,NaN
2123,af8b012e-4144-5846-b9f2-98a7d18445dd,Pass,AS Monaco,Denis Lemi Zakaria Lako Lado,55221385-bc94-510f-a39b-6613e6ac14f8,NaN,NaN
2131,1b593d33-de44-5682-8a11-a6755eee7674,Pass,AS Monaco,Eric Jeremy Edgar Dier,89389423-3218-53e6-8a20-ecff5711e40b,NaN,NaN
2261,ead1bd14-7c46-56f4-91f1-4134af40ff68,Pass,AS Monaco,Denis Lemi Zakaria Lako Lado,da66312b-f6c0-5768-b65a-654f2799fc12,NaN,NaN
2676,1c195e71-1a23-5e4b-bbc1-87ee701877eb,Pass,AS Monaco,Lamine Camara,3e53f00c-8fbf-53dd-ba23-74294d5496d9,NaN,NaN
2797,7b2bcd8c-13b1-5096-afb8-d2c52bcdfe88,Pass,Le Havre,Yassine Kechta,fc0074e9-ef27-5be3-a316-399a03f4ce77,NaN,NaN
3026,5b26b7ea-afe5-5696-b3d6-3e8ff82da9c0,Pass,Le Havre,Yassine Kechta,242aa83d-abf7-547a-8fc0-fd29bdd43319,NaN,NaN


In [32]:

# 1. Nombre de tirs dans les CSV bruts
print("===== VOLUME DES DONNÉES BRUTES =====")

print(
    raw_df["event_type_name"].value_counts().head(15)
)

raw_shots = raw_df[
    raw_df["event_type_name"].str.lower() == "shot"
].copy()

print("\nNombre de tirs bruts :", len(raw_shots))

print(
    "Tirs avec xG renseigné :",
    raw_shots["statsbomb_xg"].notna().sum()
)

print(
    "xG moyen :",
    raw_shots["statsbomb_xg"].mean()
)

# 2. Vérification des doublons
print("\n===== IDENTIFIANTS =====")

print("Lignes brutes :", len(raw_df))
print("IDs uniques :", raw_df["id"].nunique())
print("IDs manquants :", raw_df["id"].isna().sum())

# 3. Comparaison avec DuckDB
db_events = connection.execute("""
    SELECT event_id
    FROM events
""").fetchdf()

raw_ids = set(raw_df["id"].dropna())
db_ids = set(db_events["event_id"].dropna())

print("\n===== CORRESPONDANCE DUCKDB / CSV =====")

print("Événements DuckDB :", len(db_ids))
print("IDs communs :", len(raw_ids & db_ids))
print("IDs uniquement dans les CSV :", len(raw_ids - db_ids))
print("IDs uniquement dans DuckDB :", len(db_ids - raw_ids))

# 4. Vérifier les passes préparant un tir
assists = raw_df[
    (raw_df["event_type_name"] == "Pass")
    & raw_df["assisted_shot_id"].notna()
].copy()

print("\n===== PASSES ASSOCIÉES AUX TIRS =====")

print("Passes associées :", len(assists))
print(
    "Tirs associés existants :",
    assists["assisted_shot_id"].isin(raw_ids).sum()
)

display(
    assists[
        [
            "id",
            "match_id",
            "team_name",
            "player_name",
            "assisted_shot_id"
        ]
    ].head(10)
)


===== VOLUME DES DONNÉES BRUTES =====
event_type_name
Pass              33772
Ball Receipt*     29564
Carries           26926
Pressure          14267
Shot              13620
Ball Recovery      3329
Duel               2118
Block              1453
Tactical Shift     1304
Clearance          1191
Goal Keeper        1143
Foul Committed      898
Dribble             886
Dispossessed        860
Miscontrol          856
Name: count, dtype: int64

Nombre de tirs bruts : 13620
Tirs avec xG renseigné : 13620
xG moyen : 0.09818496355439794

===== IDENTIFIANTS =====
Lignes brutes : 136332
IDs uniques : 121656
IDs manquants : 0

===== CORRESPONDANCE DUCKDB / CSV =====
Événements DuckDB : 121656
IDs communs : 121656
IDs uniquement dans les CSV : 0
IDs uniquement dans DuckDB : 0

===== PASSES ASSOCIÉES AUX TIRS =====
Passes associées : 679
Tirs associés existants : 679


,id,match_id,team_name,player_name,assisted_shot_id
166,cb5ab9e0-b82b-57fa-b216-8af4085ed558,3999407,AS Monaco,Aleksandr Golovin,9116c4b9-26a9-5919-ba65-2e05e085aab9
498,ab5fab03-ab33-51fe-a1b1-6eab4e37b771,3999407,AS Monaco,Maghnes Akliouche,ba930b7e-83b1-5d71-a2ca-4137223103a5
1343,6a936571-8b97-5b7c-9533-3cb03a638149,3999407,AS Monaco,Lamine Camara,2761d763-7064-5e90-b817-67709757ae09
1514,60c99230-5e76-5d61-9744-70575b2fc390,3999407,Le Havre,Arouna Sangante,ec23a4a4-9cdf-515c-b5a7-cf5b55e2d1a6
2123,af8b012e-4144-5846-b9f2-98a7d18445dd,3999407,AS Monaco,Denis Lemi Zakaria Lako Lado,55221385-bc94-510f-a39b-6613e6ac14f8
2131,1b593d33-de44-5682-8a11-a6755eee7674,3999407,AS Monaco,Eric Jeremy Edgar Dier,89389423-3218-53e6-8a20-ecff5711e40b
2261,ead1bd14-7c46-56f4-91f1-4134af40ff68,3999407,AS Monaco,Denis Lemi Zakaria Lako Lado,da66312b-f6c0-5768-b65a-654f2799fc12
2676,1c195e71-1a23-5e4b-bbc1-87ee701877eb,3999407,AS Monaco,Lamine Camara,3e53f00c-8fbf-53dd-ba23-74294d5496d9
2797,7b2bcd8c-13b1-5096-afb8-d2c52bcdfe88,3999407,Le Havre,Yassine Kechta,fc0074e9-ef27-5be3-a316-399a03f4ce77
3026,5b26b7ea-afe5-5696-b3d6-3e8ff82da9c0,3999407,Le Havre,Yassine Kechta,242aa83d-abf7-547a-8fc0-fd29bdd43319


In [33]:
# Conserver les colonnes utiles à l'analyse
columns_to_keep = [
    "id",
    "match_id",
    "event_type_name",
    "team_id",
    "team_name",
    "player_id",
    "player_name",
    "location_x",
    "location_y",
    "end_location_x",
    "end_location_y",
    "statsbomb_xg",
    "outcome_name",
    "body_part_name",
    "technique_name",
    "assisted_shot_id",
    "shot_shot_assist",
    "shot_goal_assist",
    "pass_recipient_id",
    "pass_recipient_name",
    "pass_cross",
    "pass_cut_back",
    "pass_switch"
]

# Garder uniquement les colonnes réellement présentes
columns_to_keep = [
    col for col in columns_to_keep
    if col in raw_df.columns
]

enriched_df = raw_df[columns_to_keep].copy()

# Renommer l'identifiant
enriched_df = enriched_df.rename(
    columns={
        "id": "event_id",
        "event_type_name": "type"
    }
)

# Enregistrer temporairement les données dans DuckDB
connection.register(
    "enriched_dataframe",
    enriched_df
)

# Créer la nouvelle table
connection.execute("""
    CREATE OR REPLACE TABLE events_enriched AS
    SELECT *
    FROM enriched_dataframe
""")

connection.unregister("enriched_dataframe")

# Vérifier le résultat
result = connection.execute("""
    SELECT COUNT(*)
    FROM events_enriched
""").fetchone()[0]

print("Événements enrichis :", result)

print(
    connection.execute(
        "DESCRIBE events_enriched"
    ).fetchdf()
)

connection.close()


Événements enrichis : 136332
            column_name column_type null   key default extra
0              event_id     VARCHAR  YES  None    None  None
1              match_id      BIGINT  YES  None    None  None
2                  type     VARCHAR  YES  None    None  None
3               team_id      BIGINT  YES  None    None  None
4             team_name     VARCHAR  YES  None    None  None
5             player_id      DOUBLE  YES  None    None  None
6           player_name     VARCHAR  YES  None    None  None
7            location_x      DOUBLE  YES  None    None  None
8            location_y      DOUBLE  YES  None    None  None
9        end_location_x      DOUBLE  YES  None    None  None
10       end_location_y      DOUBLE  YES  None    None  None
11         statsbomb_xg      DOUBLE  YES  None    None  None
12         outcome_name     VARCHAR  YES  None    None  None
13       body_part_name     VARCHAR  YES  None    None  None
14       technique_name     VARCHAR  YES  None    None  